# MDR Flowchart Pipeline for Kaggle

This notebook runs the original script-style pipeline in Kaggle.

Flow:
`PDF → detect charts → segment nodes → infer edges → OCR node/label text with GLM-OCR → JSON + Mermaid + debug overlays`

Use `DRY_RUN=True` first to test geometry without loading GLM-OCR.

**New in this version:** soft raster fallback keeps large colored embedded diagram images that fail strict line-art checks, then lets graph validation decide.

Update: this version adds `nodes_from_soft_raster()` and background-aware edge masks so dark/colored embedded flowcharts can be parsed instead of only detected.

In [11]:
# Cell 1: Install dependencies
# Run this once. If GLM-OCR gives `glm_ocr` architecture errors, restart the Kaggle session
# after this install cell, then run from Cell 2 onward.

!pip install -q PyMuPDF opencv-python-headless numpy pillow torch pandas tqdm
!pip install -q git+https://github.com/huggingface/transformers.git accelerate sentencepiece safetensors

print("Install complete. If transformers was upgraded, restart Kaggle session once before running the remaining cells.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Install complete. If transformers was upgraded, restart Kaggle session once before running the remaining cells.


In [12]:
# Cell 2: Config — change these only

PDF_PATH = "/kaggle/input/datasets/boopaland3v/sam-data/The_European_Medical_Device_Regulation-SECURED.pdf"
OUT_DIR = "/kaggle/working/mdr_flowchart_build"

# Use DRY_RUN=True first. It skips GLM-OCR and tests chart detection / nodes / edges only.
DRY_RUN = True

# Turn debug overlays on. Green=nodes, red=connectors, blue=labels.
DEBUG = True

# Optional: process only first N detected charts while debugging.
# Use None for all.
MAX_CHARTS = None

In [13]:
# Cell 3: Imports, tunables, data models

from __future__ import annotations

import json
import os
import time
from collections import Counter
from dataclasses import dataclass, field, asdict
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception as e:
    raise RuntimeError("PyMuPDF import failed. Run Cell 1 first, then restart if needed.") from e


# ----------------------------------------------------------------------------
# Tunables
# ----------------------------------------------------------------------------
MIN_COVERAGE, MAX_COVERAGE = 0.15, 0.90
MIN_DIAGONALS, MIN_CLOSED = 30, 8
MIN_RASTER_PX = 700

# Soft raster fallback: keep large colored embedded diagrams that fail strict line-art checks.
# This catches flowcharts stored as colored raster images instead of vector drawings.
ENABLE_SOFT_RASTER_CANDIDATES = True
SOFT_RASTER_MIN_COVERAGE = 0.10
SOFT_RASTER_MAX_COVERAGE = 0.95
SOFT_RASTER_MIN_PX = 500
MAX_LINEART_COLORS, MIN_WHITE_FRAC = 64, 0.35
VECTOR_DPI = 400
PAD_PT = 6

INK_THRESHOLD = 245
NODE_DILATE = 3
ARROWHEAD_RATIO = 1.6
ARROWHEAD_SEARCH = 15
BRIDGE_DILATE = 21
MIN_LABEL_AREA = 20
MIN_CROP_H = 224

MODEL_PATH = "zai-org/GLM-OCR"
OCR_PROMPT = "Text Recognition:"
OCR_BATCH = 8


@dataclass
class Node:
    id: str
    bbox: tuple
    shape: str
    text: str = ""


@dataclass
class Chart:
    page: int
    kind: str
    image_path: str
    clip: list | None = None
    dpi: int | None = None
    nodes: list = field(default_factory=list)
    edges: list = field(default_factory=list)
    confident: bool = True


TIMING = {
    "start": None,
    "end": None,
    "stages": {},
    "charts": {}
}

def tick(name):
    TIMING["stages"].setdefault(name, 0.0)
    return time.perf_counter()

def tock(name, t0):
    TIMING["stages"][name] = TIMING["stages"].get(name, 0.0) + (time.perf_counter() - t0)

def fmt(sec):
    sec = float(sec or 0)
    if sec < 60:
        return f"{sec:.2f}s"
    return f"{sec/60:.2f}m"

In [14]:
# Cell 4: Stage 1 — detect and extract chart candidates from PDF

def _is_line_art(pix: fitz.Pixmap) -> bool:
    """Line art = few unique colors + mostly white. Photographs fail both."""
    if pix.n > 3 or pix.alpha:
        pix = fitz.Pixmap(fitz.csRGB, pix)

    s, n = pix.samples, pix.n
    step = max(1, (len(s) // n) // 20000)
    colors, white, total = Counter(), 0, 0

    for i in range(0, len(s) - n, n * step):
        px = tuple(s[i:i + 3])
        colors[px] += 1
        total += 1
        if min(px) > 235:
            white += 1

    return len(colors) <= MAX_LINEART_COLORS and white / max(total, 1) >= MIN_WHITE_FRAC


def _vector_signature(page, clip):
    """Count diagonal segments and closed/filled paths inside a drawing cluster."""
    diagonals = closed = 0

    for d in page.get_drawings():
        if not fitz.Rect(d["rect"]).intersects(clip):
            continue

        if d.get("closePath") or d.get("fill"):
            closed += 1

        for it in d["items"]:
            if it[0] == "l":
                p1, p2 = it[1], it[2]
                if abs(p2.x - p1.x) > 1.0 and abs(p2.y - p1.y) > 1.0:
                    diagonals += 1

    return diagonals, closed


def detect_charts(pdf_path, out_dir) -> list[Chart]:
    """Detect vector/raster flowchart candidates and save each chart image."""
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    charts: list[Chart] = []

    for pno in tqdm(range(doc.page_count), desc="Stage 1: detect charts"):
        page = doc[pno]
        parea = page.rect.get_area()

        # Raster charts: embedded images.
        # Strong path: strict line-art image -> raster.
        # Soft fallback: large colored/low-white diagram image -> soft_raster, then graph validation decides.
        for info in page.get_image_info(xrefs=True):
            xref = info["xref"]

            if xref == 0:
                continue

            img_w = info.get("width", 0)
            img_h = info.get("height", 0)

            if img_w < SOFT_RASTER_MIN_PX or img_h < SOFT_RASTER_MIN_PX:
                continue

            bbox = fitz.Rect(info["bbox"])
            cov = bbox.get_area() / parea

            strong_coverage = MIN_COVERAGE <= cov <= MAX_COVERAGE
            soft_coverage = SOFT_RASTER_MIN_COVERAGE <= cov <= SOFT_RASTER_MAX_COVERAGE

            if not strong_coverage and not soft_coverage:
                continue

            try:
                pix = fitz.Pixmap(doc, xref)

                if pix.n > 3 or pix.alpha:
                    pix = fitz.Pixmap(fitz.csRGB, pix)

                is_line_art = _is_line_art(pix)

                # Existing strict behavior: strong raster candidate.
                if strong_coverage and is_line_art:
                    img_path = os.path.join(out_dir, f"chart_p{pno + 1:03d}_raster.png")
                    pix.save(img_path)
                    charts.append(Chart(page=pno + 1, kind="raster", image_path=img_path))
                    continue

                # New safe fallback: do not discard large colored embedded diagrams immediately.
                # These are marked as soft_raster and later accepted/rejected by node+edge validation.
                if ENABLE_SOFT_RASTER_CANDIDATES and soft_coverage:
                    img_path = os.path.join(out_dir, f"chart_p{pno + 1:03d}_soft_raster.png")
                    pix.save(img_path)
                    charts.append(Chart(page=pno + 1, kind="soft_raster", image_path=img_path))

            except Exception as e:
                print(f"Raster extraction failed on page {pno + 1}: {e}")

        # Vector charts: drawing clusters with diagonals + closed shapes
        try:
            drawings = page.get_drawings()
        except Exception:
            drawings = []

        if len(drawings) < 50:
            continue

        try:
            clusters = page.cluster_drawings(drawings=drawings)
        except Exception:
            clusters = []

        for cluster in clusters:
            rect = fitz.Rect(cluster)
            cov = rect.get_area() / parea

            if not (MIN_COVERAGE <= cov <= MAX_COVERAGE):
                continue

            diag, closed = _vector_signature(page, rect)

            # Tables die here because they usually have zero diagonals.
            if diag < MIN_DIAGONALS or closed < MIN_CLOSED:
                continue

            clip = (rect + (-PAD_PT, -PAD_PT, PAD_PT, PAD_PT)) & page.rect
            pix = page.get_pixmap(dpi=VECTOR_DPI, clip=clip, alpha=False)
            img_path = os.path.join(out_dir, f"chart_p{pno + 1:03d}_vector.png")
            pix.save(img_path)

            charts.append(
                Chart(
                    page=pno + 1,
                    kind="vector",
                    image_path=img_path,
                    clip=[clip.x0, clip.y0, clip.x1, clip.y1],
                    dpi=VECTOR_DPI
                )
            )

    doc.close()
    charts.sort(key=lambda c: (c.page, c.kind))
    return charts

In [15]:
# Cell 5: Stage 2 — node segmentation

def _path_points(d):
    pts = []

    for it in d["items"]:
        if it[0] == "l":
            pts += [(it[1].x, it[1].y), (it[2].x, it[2].y)]
        elif it[0] == "re":
            r = it[1]
            pts += [(r.x0, r.y0), (r.x1, r.y0), (r.x1, r.y1), (r.x0, r.y1)]
        elif it[0] == "c":
            pts += [(p.x, p.y) for p in it[1:5]]
        elif it[0] == "qu":
            pts += [(p.x, p.y) for p in it[1]]

    return pts


def nodes_from_vector(pdf_path, page_no, clip, dpi, img_shape):
    """
    Exact node polygons from vector path data.
    Uses polygon-area / bbox-area to reject arrow ribbons.
    """
    FILL_RATIO_MIN = 0.25

    doc = fitz.open(pdf_path)
    page = doc[page_no - 1]
    scale = dpi / 72.0
    mask = np.zeros(img_shape[:2], np.uint8)
    raw = []

    for d in page.get_drawings():
        r = fitz.Rect(d["rect"])

        if r.width < 30 or r.height < 15:
            continue
        if r.width > 500 and r.height > 400:
            continue

        pts = _path_points(d)
        if len(pts) < 3:
            continue

        poly = np.array(pts, np.float32)
        fill_ratio = abs(cv2.contourArea(poly)) / max(r.width * r.height, 1)

        if fill_ratio < FILL_RATIO_MIN:
            continue

        items = d["items"]
        diag = any(
            it[0] == "l"
            and abs(it[2].x - it[1].x) > 1
            and abs(it[2].y - it[1].y) > 1
            for it in items
        )
        curve = any(it[0] == "c" for it in items)

        shape = "diamond" if diag else ("stadium" if curve else "rect")

        px = np.array(
            [[(x - clip[0]) * scale, (y - clip[1]) * scale] for x, y in pts],
            np.int32
        )

        cv2.fillPoly(mask, [px], 255)
        raw.append((r, shape))

    doc.close()

    # Merge duplicates
    merged = []
    for r, shape in sorted(raw, key=lambda t: (t[0].y0, t[0].x0)):
        hit = None

        for m in merged:
            inter = (r & m["rect"]).get_area()
            union = r.get_area() + m["rect"].get_area() - inter

            if union > 0 and inter / union > 0.55:
                hit = m
                break

        if hit:
            hit["rect"] |= r
            if shape == "diamond":
                hit["shape"] = "diamond"
        else:
            merged.append({"rect": +r, "shape": shape})

    nodes = []
    for i, m in enumerate(merged):
        r = m["rect"]
        bbox = (
            int((r.x0 - clip[0]) * scale),
            int((r.y0 - clip[1]) * scale),
            int(r.width * scale),
            int(r.height * scale)
        )
        nodes.append(Node(id=f"n{i}", bbox=bbox, shape=m["shape"]))

    return nodes, mask, True


def nodes_from_raster(img_bgr):
    """
    Morphology-based node detection.
    Kernel wider than stroke: arrows/text vanish, node bodies survive.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    ink = ((gray < INK_THRESHOLD).astype(np.uint8)) * 255
    H, W = ink.shape

    # Fill enclosed node interiors.
    inv = cv2.bitwise_not(ink)
    ff = inv.copy()
    cv2.floodFill(ff, np.zeros((H + 2, W + 2), np.uint8), (0, 0), 0)
    filled = cv2.bitwise_or(ink, ff)

    k = max(7, int(min(img_bgr.shape[:2]) * 0.006)) | 1
    solid = cv2.morphologyEx(filled, cv2.MORPH_OPEN, np.ones((k, k), np.uint8))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(solid, 8)
    area_min = H * W * 0.0008

    keep = []
    mask = np.zeros(ink.shape, np.uint8)
    confident = True

    for i in range(1, n):
        x, y, w, h, area = stats[i]

        if area < area_min or w < 25 or h < 25:
            continue

        if area > 0.25 * H * W:
            confident = False

        mask[lab == i] = 255
        roi = (lab[y:y + h, x:x + w] == i).astype(np.uint8) * 255

        cnts, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        shape = "rect"

        if cnts:
            c = max(cnts, key=cv2.contourArea)
            ap = cv2.approxPolyDP(c, 0.03 * cv2.arcLength(c, True), True)
            fill = cv2.contourArea(c) / max(w * h, 1)

            if len(ap) == 4 and fill < 0.62:
                shape = "diamond"
            elif len(ap) > 6 and fill > 0.85:
                shape = "stadium"

        keep.append(((int(x), int(y), int(w), int(h)), shape))

    keep.sort(key=lambda t: (t[0][1], t[0][0]))
    nodes = [Node(id=f"n{i}", bbox=b, shape=s) for i, (b, s) in enumerate(keep)]

    return nodes, mask, confident

# ----------------------------------------------------------------------------
# Soft raster node extraction
# ----------------------------------------------------------------------------
def _merge_node_boxes(boxes, iou_threshold=0.25):
    """
    Merge duplicate / overlapping node boxes produced by soft raster detection.
    Keeps the original pipeline untouched; used only for kind='soft_raster'.
    """
    merged = []

    for bbox, shape in sorted(boxes, key=lambda item: (item[0][1], item[0][0])):
        x, y, w, h = bbox
        hit = None

        for item in merged:
            mx, my, mw, mh = item["bbox"]

            ix0 = max(x, mx)
            iy0 = max(y, my)
            ix1 = min(x + w, mx + mw)
            iy1 = min(y + h, my + mh)

            inter = max(0, ix1 - ix0) * max(0, iy1 - iy0)
            union = (w * h) + (mw * mh) - inter
            iou = inter / max(union, 1)

            contains = (
                (x >= mx - 4 and y >= my - 4 and x + w <= mx + mw + 4 and y + h <= my + mh + 4)
                or
                (mx >= x - 4 and my >= y - 4 and mx + mw <= x + w + 4 and my + mh <= y + h + 4)
            )

            if iou > iou_threshold or contains:
                hit = item
                break

        if hit:
            mx, my, mw, mh = hit["bbox"]
            nx = min(x, mx)
            ny = min(y, my)
            nx2 = max(x + w, mx + mw)
            ny2 = max(y + h, my + mh)
            hit["bbox"] = (nx, ny, nx2 - nx, ny2 - ny)
            if shape == "diamond":
                hit["shape"] = "diamond"
        else:
            merged.append({"bbox": bbox, "shape": shape})

    return [(item["bbox"], item["shape"]) for item in merged]


def nodes_from_soft_raster(img_bgr):
    """
    Soft-raster node detector for colored / dark-background embedded flowcharts.

    Why this exists:
    - nodes_from_raster() assumes black ink on mostly white background.
    - FDA-style embedded flowchart images can have a dark/black background and colored/white nodes.
    - So this function detects foreground node bodies using background-aware thresholding,
      then removes thin arrows/text with morphology.

    It is used only when chart.kind == 'soft_raster'.
    """
    H, W = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Estimate page/chart background from image border.
    border_px = max(3, min(H, W) // 100)
    border = np.concatenate([
        gray[:border_px, :].ravel(),
        gray[-border_px:, :].ravel(),
        gray[:, :border_px].ravel(),
        gray[:, -border_px:].ravel(),
    ])
    bg = float(np.median(border))

    # If background is dark, foreground objects are bright/colored.
    # If background is light, foreground objects are dark lines/fills.
    if bg < 80:
        foreground = (gray > max(25, bg + 25)).astype(np.uint8) * 255
    else:
        foreground = (gray < min(245, bg - 10)).astype(np.uint8) * 255

    # Remove thin arrows/text; keep filled/large node bodies.
    k = max(5, int(min(H, W) * 0.008)) | 1
    solid = cv2.morphologyEx(foreground, cv2.MORPH_OPEN, np.ones((k, k), np.uint8))
    solid = cv2.morphologyEx(solid, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(solid, 8)

    candidates = []
    node_mask = np.zeros((H, W), np.uint8)

    min_area = H * W * 0.001
    min_w = max(30, int(W * 0.035))
    min_h = max(20, int(H * 0.018))

    for i in range(1, n):
        x, y, w, h, area = stats[i]

        if area < min_area or w < min_w or h < min_h:
            continue

        # Avoid full-page / giant merged background-like components.
        if area > 0.35 * H * W:
            continue

        aspect = w / max(h, 1)
        if aspect > 8 or aspect < 0.12:
            continue

        roi = (lab[y:y + h, x:x + w] == i).astype(np.uint8) * 255
        contours, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        shape = "rect"
        if contours:
            contour = max(contours, key=cv2.contourArea)
            peri = cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, 0.035 * peri, True)
            fill = cv2.contourArea(contour) / max(w * h, 1)

            if len(approx) == 4 and fill < 0.70:
                shape = "diamond"
            elif len(approx) > 6 and fill > 0.70:
                shape = "stadium"

        candidates.append(((int(x), int(y), int(w), int(h)), shape))

    candidates = _merge_node_boxes(candidates, iou_threshold=0.25)
    candidates = sorted(candidates, key=lambda item: (item[0][1], item[0][0]))

    for bbox, _shape in candidates:
        x, y, w, h = bbox
        # Use bbox-filled mask so edge extraction can cleanly subtract node bodies.
        cv2.rectangle(node_mask, (x, y), (x + w, y + h), 255, -1)

    nodes = [Node(id=f"n{i}", bbox=bbox, shape=shape) for i, (bbox, shape) in enumerate(candidates)]
    confident = len(nodes) >= 3

    return nodes, node_mask, confident


In [16]:
# Cell 6: Stage 3 — connector/edge extraction

# ----------------------------------------------------------------------------
# Background-aware ink/foreground mask
# ----------------------------------------------------------------------------
def foreground_ink_mask(img_bgr):
    """
    Returns the connector/text/ink mask for both:
    - normal white-background diagrams: dark ink = foreground
    - dark-background soft raster diagrams: bright/colored objects = foreground

    This keeps the existing edge extraction logic intact while fixing dark-background
    embedded flowcharts where the old gray < 245 rule treated the whole background as ink.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape[:2]

    border_px = max(3, min(H, W) // 100)
    border = np.concatenate([
        gray[:border_px, :].ravel(),
        gray[-border_px:, :].ravel(),
        gray[:, :border_px].ravel(),
        gray[:, -border_px:].ravel(),
    ])
    bg = float(np.median(border))

    if bg < 80:
        # dark background: arrows/nodes/text are brighter than background
        mask = (gray > max(25, bg + 25)).astype(np.uint8) * 255
    else:
        # light background: arrows/nodes/text are darker than background
        mask = (gray < INK_THRESHOLD).astype(np.uint8) * 255

    return mask


def _contacts(comp_mask, node_masks_dilated):
    out = []

    for node_id, node_mask in node_masks_dilated.items():
        hit = cv2.bitwise_and(comp_mask, node_mask)

        if cv2.countNonZero(hit) < 3:
            continue

        out.append((node_id, hit))

    return out


def _per_node_masks(nodes, nm_dil, shape):
    per = {}

    for node in nodes:
        x, y, w, h = node.bbox
        m = np.zeros(shape, np.uint8)

        y0, y1 = max(y - 4, 0), min(y + h + 4, shape[0])
        x0, x1 = max(x - 4, 0), min(x + w + 4, shape[1])

        m[y0:y1, x0:x1] = nm_dil[y0:y1, x0:x1]
        per[node.id] = cv2.dilate(m, np.ones((13, 13), np.uint8))

    return per


def _classify_connector(comp_mask, per_node, dist, real_ink):
    """
    Split a connector's contacts into sources/targets.
    Arrowhead is usually thicker than the source line.
    """
    touched = _contacts(comp_mask, per_node)

    if len(touched) < 2:
        return None

    peaks = []

    for node_id, hit in touched:
        real = cv2.bitwise_and(hit, real_ink)
        ys, xs = np.nonzero(real if cv2.countNonZero(real) else hit)

        if len(xs) == 0:
            continue

        cx, cy = float(xs.mean()), float(ys.mean())

        if not cv2.countNonZero(real):
            peaks.append((node_id, 0.0, (cx, cy)))
            continue

        y0, y1 = max(int(cy) - ARROWHEAD_SEARCH, 0), int(cy) + ARROWHEAD_SEARCH
        x0, x1 = max(int(cx) - ARROWHEAD_SEARCH, 0), int(cx) + ARROWHEAD_SEARCH

        win = dist[y0:y1, x0:x1]
        peaks.append((node_id, float(win.max()) if win.size else 0.0, (cx, cy)))

    if len(peaks) < 2:
        return None

    hi = max(p for _, p, _ in peaks)
    lo = min(p for _, p, _ in peaks)

    if hi > ARROWHEAD_RATIO * max(lo, 0.5):
        targets = [n for n, p, _ in peaks if p >= 0.75 * hi]
        sources = [n for n, p, _ in peaks if n not in targets]
    else:
        # fallback: reading order
        peaks.sort(key=lambda t: (t[2][1], t[2][0]))
        sources = [peaks[0][0]]
        targets = [n for n, _, _ in peaks[1:]]

    if not sources:
        sources = [n for n, _, _ in peaks if n not in targets]

    return sources, targets


def group_labels(label_blobs, shape):
    """Merge adjacent word blobs into label boxes."""
    if not label_blobs:
        return []

    m = np.zeros(shape[:2], np.uint8)

    for x, y, w, h in label_blobs:
        cv2.rectangle(m, (x, y), (x + w, y + h), 255, -1)

    m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 9)))
    n, _, stats, _ = cv2.connectedComponentsWithStats(m, 8)

    out = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]

        if area < MIN_LABEL_AREA:
            continue

        out.append((int(x), int(y), int(w), int(h)))

    return out


def extract_edges(img_bgr, nodes, node_mask, debug=None):
    """
    Connector layer = dark ink - node polygons.
    Two-pass logic bridges short Yes/No label gaps.
    """
    H, W = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    ink = foreground_ink_mask(img_bgr)

    nm_dil = cv2.dilate(node_mask, np.ones((NODE_DILATE * 2 + 1,) * 2, np.uint8))
    conn = cv2.bitwise_and(ink, cv2.bitwise_not(nm_dil))
    per_node = _per_node_masks(nodes, nm_dil, ink.shape)
    dist = cv2.distanceTransform(conn, cv2.DIST_L2, 3)

    # Pass 1: fragments touching fewer than 2 nodes are likely labels/notes
    ncomp, lab, stats, _ = cv2.connectedComponentsWithStats(conn, 8)

    fragments = []
    for i in range(1, ncomp):
        if stats[i, cv2.CC_STAT_AREA] < MIN_LABEL_AREA:
            continue

        comp = (lab == i).astype(np.uint8) * 255

        if len(_contacts(comp, per_node)) < 2:
            fragments.append(
                (
                    int(stats[i, cv2.CC_STAT_LEFT]),
                    int(stats[i, cv2.CC_STAT_TOP]),
                    int(stats[i, cv2.CC_STAT_WIDTH]),
                    int(stats[i, cv2.CC_STAT_HEIGHT])
                )
            )

    label_boxes = group_labels(fragments, img_bgr.shape)

    # Pass 2: bridge short labels like Yes/No
    bridge = np.zeros(ink.shape, np.uint8)

    for x, y, w, h in label_boxes:
        if w < 0.12 * W and h < 0.06 * H:
            cv2.rectangle(bridge, (x, y), (x + w, y + h), 255, -1)

    bridge = cv2.dilate(bridge, np.ones((BRIDGE_DILATE, BRIDGE_DILATE), np.uint8))
    bridge = cv2.bitwise_and(bridge, cv2.bitwise_not(nm_dil))
    conn2 = cv2.bitwise_or(conn, bridge)

    ncomp, lab, stats, _ = cv2.connectedComponentsWithStats(conn2, 8)

    edges = []
    for i in range(1, ncomp):
        if stats[i, cv2.CC_STAT_AREA] < MIN_LABEL_AREA:
            continue

        comp = (lab == i).astype(np.uint8) * 255
        res = _classify_connector(comp, per_node, dist, conn)

        if res is None:
            continue

        sources, targets = res

        for sid in sources:
            for tid in targets:
                if sid != tid:
                    edges.append(
                        {
                            "source": sid,
                            "target": tid,
                            "comp": int(i),
                            "label": ""
                        }
                    )

    if debug is not None:
        debug["conn"] = conn2
        debug["labels"] = label_boxes

    return edges, label_boxes, lab


def attach_labels(edges, labels, nodes, comp_lab):
    """
    Label box lies inside connector component after bridging.
    Attach OCR label to nearest matching edge component.
    """
    ncent = {
        n.id: (n.bbox[0] + n.bbox[2] / 2, n.bbox[1] + n.bbox[3] / 2)
        for n in nodes
    }

    for (lx, ly, lw, lh), text in labels:
        if not text:
            continue

        patch = comp_lab[ly:ly + lh, lx:lx + lw]
        ids, counts = np.unique(patch[patch > 0], return_counts=True)

        if not len(ids):
            continue

        comp_id = int(ids[counts.argmax()])
        candidates = [e for e in edges if e["comp"] == comp_id and not e["label"]]

        if not candidates:
            continue

        cx, cy = lx + lw / 2, ly + lh / 2

        candidates.sort(
            key=lambda e: (
                (cx - ncent[e["source"]][0]) ** 2
                + (cy - ncent[e["source"]][1]) ** 2
            )
        )

        candidates[0]["label"] = text

    return edges

In [17]:
# Cell 7: Stage 4 — GLM-OCR text reading for node crops and edge labels

class GlmOcr:
    def __init__(self, model_path=MODEL_PATH, dry_run=False):
        self.dry_run = dry_run

        if dry_run:
            print("GLM-OCR dry-run mode: OCR will return crop filenames only.")
            return

        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        self.torch = torch
        self.processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
        self.model = AutoModelForImageTextToText.from_pretrained(
            pretrained_model_name_or_path=model_path,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True,
        )
        self.model.eval()

        tok = getattr(self.processor, "tokenizer", None)
        if tok is not None:
            tok.padding_side = "left"

    def _msg(self, path):
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image", "url": str(path)},
                    {"type": "text", "text": OCR_PROMPT},
                ],
            }
        ]

    def _msg_pil(self, path):
        image = Image.open(path).convert("RGB")
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": OCR_PROMPT},
                ],
            }
        ]

    def read_one(self, path, max_new_tokens=384):
        if self.dry_run:
            return f"<{os.path.basename(path)}>"

        with self.torch.inference_mode():
            try:
                msg = self._msg(path)
                inputs = self.processor.apply_chat_template(
                    msg,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt",
                ).to(self.model.device)
            except Exception:
                msg = self._msg_pil(path)
                inputs = self.processor.apply_chat_template(
                    msg,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt",
                ).to(self.model.device)

            inputs.pop("token_type_ids", None)

            ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

            cut = inputs["input_ids"].shape[1]
            return self.processor.decode(
                ids[0][cut:],
                skip_special_tokens=True
            ).strip()

    def read(self, paths, max_new_tokens=384):
        return [self.read_one(p, max_new_tokens=max_new_tokens) for p in paths]


def write_crop(img, bbox, path, inset=0):
    x, y, w, h = bbox
    x, y = x + inset, y + inset
    w, h = max(w - 2 * inset, 1), max(h - 2 * inset, 1)

    crop = img[max(y, 0):y + h, max(x, 0):x + w]

    if crop.size == 0:
        return False

    if crop.shape[0] < MIN_CROP_H:
        scale = MIN_CROP_H / crop.shape[0]
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_LANCZOS4)

    crop = cv2.copyMakeBorder(
        crop,
        16,
        16,
        16,
        16,
        cv2.BORDER_CONSTANT,
        value=(255, 255, 255),
    )

    cv2.imwrite(path, crop)
    return True

In [18]:
# Cell 8: Output helpers and main run function

def to_mermaid(chart: Chart) -> str:
    wrap = {
        "diamond": ("{", "}"),
        "stadium": ("([", "])"),
        "rect": ("[", "]")
    }

    lines = [f"%% page {chart.page} ({chart.kind})", "flowchart TD"]

    for n in chart.nodes:
        open_br, close_br = wrap.get(n.shape, ("[", "]"))
        text = n.text.replace("\n", " ").replace('"', "'").strip() or n.id
        lines.append(f'    {n.id}{open_br}"{text}"{close_br}')

    for e in chart.edges:
        arrow = f'-- "{e.get("label", "")}" -->' if e.get("label") else "-->"
        lines.append(f'    {e["source"]} {arrow} {e["target"]}')

    return "\n".join(lines)


def draw_debug(img, nodes, conn, labels, path):
    overlay = img.copy()

    if conn is not None:
        overlay[conn > 0] = (0, 0, 255)

    for n in nodes:
        x, y, w, h = n.bbox
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 180, 0), 3)
        cv2.putText(
            overlay,
            n.id,
            (x + 4, y + 26),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 120, 0),
            2,
        )

    for x, y, w, h in labels:
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (255, 0, 0), 2)

    cv2.imwrite(path, overlay)



def validate_graph_output(chart):
    """
    Integrated graph-quality validation.
    This does not change detection, nodes, edges, OCR, JSON, or Mermaid logic.
    It only adds final quality metadata to the same output objects.
    """
    node_count = len(chart.nodes)
    edge_count = len(chart.edges)
    edge_node_ratio = edge_count / max(node_count, 1)

    reasons = []

    if node_count < 3:
        reasons.append("too_few_nodes")

    if edge_count < 2:
        reasons.append("too_few_edges")

    if node_count >= 5 and edge_count == 0:
        reasons.append("nodes_without_edges_possible_false_positive")

    if edge_node_ratio < 0.15:
        reasons.append("very_low_edge_node_ratio")

    if chart.confident is False:
        reasons.append("node_segmentation_low_confidence")

    is_valid_flowchart_graph = (
        node_count >= 3
        and edge_count >= 2
        and edge_node_ratio >= 0.15
        and chart.confident is True
    )

    if chart.kind == "soft_raster" and not is_valid_flowchart_graph:
        reasons.append("soft_raster_candidate_needs_graph_confirmation")

    return {
        "is_valid_flowchart_graph": is_valid_flowchart_graph,
        "needs_review": not is_valid_flowchart_graph,
        "validation_reasons": reasons,
        "node_count": node_count,
        "edge_count": edge_count,
        "edge_node_ratio": round(edge_node_ratio, 4),
    }


def run_pipeline(pdf_path, out_dir, dry_run=True, debug=True, max_charts=None):
    TIMING["start"] = time.perf_counter()
    os.makedirs(out_dir, exist_ok=True)

    chart_dir = os.path.join(out_dir, "charts")
    crop_dir = os.path.join(out_dir, "crops")
    os.makedirs(crop_dir, exist_ok=True)

    t0 = tick("detect_charts")
    charts = detect_charts(pdf_path, chart_dir)
    tock("detect_charts", t0)

    if max_charts is not None:
        charts = charts[:max_charts]

    print(f"Detected {len(charts)} flowchart candidates:", [c.page for c in charts])

    t0 = tick("load_ocr")
    ocr = GlmOcr(dry_run=dry_run)
    tock("load_ocr", t0)

    all_json = []

    for ch in tqdm(charts, desc="Process charts"):
        chart_t0 = time.perf_counter()
        tag = f"p{ch.page:03d}_{ch.kind}"
        local_times = {}

        img = cv2.imread(ch.image_path)
        if img is None:
            print(f"Could not read chart image: {ch.image_path}")
            continue

        # Nodes
        s = time.perf_counter()
        if ch.kind == "vector":
            ch.nodes, node_mask, ok = nodes_from_vector(pdf_path, ch.page, ch.clip, ch.dpi, img.shape)
        elif ch.kind == "soft_raster":
            # Soft raster flowcharts can have dark/colored backgrounds.
            # Use the dedicated background-aware node extractor.
            ch.nodes, node_mask, ok = nodes_from_soft_raster(img)
        else:
            ch.nodes, node_mask, ok = nodes_from_raster(img)
        ch.confident = ok
        local_times["nodes"] = time.perf_counter() - s

        # Edges
        s = time.perf_counter()
        dbg = {}
        edges, label_boxes, comp_lab = extract_edges(img, ch.nodes, node_mask, debug=dbg)
        local_times["edges"] = time.perf_counter() - s

        # OCR crops
        s = time.perf_counter()
        chart_crop_dir = os.path.join(crop_dir, tag)
        os.makedirs(chart_crop_dir, exist_ok=True)

        node_paths, node_keep = [], []
        for node in ch.nodes:
            p = os.path.join(chart_crop_dir, f"{node.id}.png")
            if write_crop(img, node.bbox, p, inset=4):
                node_paths.append(p)
                node_keep.append(node)

        for node, text in zip(node_keep, ocr.read(node_paths)):
            node.text = text

        label_paths = []
        for j, bbox in enumerate(label_boxes):
            p = os.path.join(chart_crop_dir, f"lbl{j}.png")
            if write_crop(img, bbox, p):
                label_paths.append((bbox, p))

        label_texts = ocr.read([p for _, p in label_paths]) if label_paths else []
        labels = [(bbox, text) for (bbox, _), text in zip(label_paths, label_texts)]

        short_labels = [(bbox, text) for bbox, text in labels if len(text.strip()) <= 12]
        ch.edges = attach_labels(edges, short_labels, ch.nodes, comp_lab)

        for e in ch.edges:
            e.pop("comp", None)

        local_times["ocr"] = time.perf_counter() - s

        if debug:
            debug_path = os.path.join(out_dir, f"debug_{tag}.png")
            draw_debug(img, ch.nodes, dbg.get("conn"), label_boxes, debug_path)

        validation = validate_graph_output(ch)

        chart_json = {
            "page": ch.page,
            "kind": ch.kind,
            "image_path": ch.image_path,
            "confident": ch.confident,
            "is_valid_flowchart_graph": validation["is_valid_flowchart_graph"],
            "needs_review": validation["needs_review"],
            "validation_reasons": validation["validation_reasons"],
            "node_count": validation["node_count"],
            "edge_count": validation["edge_count"],
            "edge_node_ratio": validation["edge_node_ratio"],
            "nodes": [asdict(n) for n in ch.nodes],
            "edges": ch.edges,
        }

        with open(os.path.join(out_dir, f"{tag}.json"), "w", encoding="utf-8") as f:
            json.dump(chart_json, f, indent=2, ensure_ascii=False)

        with open(os.path.join(out_dir, f"{tag}.mmd"), "w", encoding="utf-8") as f:
            f.write(to_mermaid(ch))

        all_json.append(chart_json)

        TIMING["charts"][tag] = {
            **local_times,
            "total": time.perf_counter() - chart_t0,
            "page": ch.page,
            "kind": ch.kind,
            "nodes": validation["node_count"],
            "edges": validation["edge_count"],
            "edge_node_ratio": validation["edge_node_ratio"],
            "is_valid_flowchart_graph": validation["is_valid_flowchart_graph"],
            "needs_review": validation["needs_review"],
            "validation_reasons": ", ".join(validation["validation_reasons"]) if validation["validation_reasons"] else "",
            "confident": ch.confident,
        }

        status = "VALID" if validation["is_valid_flowchart_graph"] else "REVIEW"
        reason_text = f" | reasons={validation['validation_reasons']}" if validation["validation_reasons"] else ""
        flag = "" if ch.confident else " [LOW CONFIDENCE]"
        soft_note = " [SOFT RASTER]" if ch.kind == "soft_raster" else ""
        print(
            f"{tag}{soft_note}: {len(ch.nodes)} nodes, {len(ch.edges)} edges, "
            f"{len(label_boxes)} label blobs | {status}{flag}{reason_text}"
        )

    with open(os.path.join(out_dir, "all_graphs.json"), "w", encoding="utf-8") as f:
        json.dump(all_json, f, indent=2, ensure_ascii=False)

    TIMING["end"] = time.perf_counter()

    return charts, all_json

In [19]:
# Cell 9: Run in Kaggle

charts, all_graphs = run_pipeline(
    pdf_path=PDF_PATH,
    out_dir=OUT_DIR,
    dry_run=DRY_RUN,
    debug=DEBUG,
    max_charts=MAX_CHARTS,
)

print("\nDone.")
print("Output folder:", OUT_DIR)
print("Detected chart pages:", [c.page for c in charts])

Stage 1: detect charts:   0%|          | 0/384 [00:00<?, ?it/s]

Detected 9 flowchart candidates: [14, 15, 16, 17, 275, 336, 352, 354, 364]
GLM-OCR dry-run mode: OCR will return crop filenames only.


Process charts:   0%|          | 0/9 [00:00<?, ?it/s]

p014_vector: 18 nodes, 19 edges, 29 label blobs | VALID
p015_raster: 23 nodes, 20 edges, 16 label blobs | VALID
p016_raster: 17 nodes, 17 edges, 15 label blobs | VALID
p017_raster: 20 nodes, 19 edges, 9 label blobs | VALID
p275_soft_raster [SOFT RASTER]: 3 nodes, 1 edges, 2 label blobs | REVIEW | reasons=['too_few_edges', 'soft_raster_candidate_needs_graph_confirmation']
p336_soft_raster [SOFT RASTER]: 3 nodes, 0 edges, 5 label blobs | REVIEW | reasons=['too_few_edges', 'very_low_edge_node_ratio', 'soft_raster_candidate_needs_graph_confirmation']
p352_soft_raster [SOFT RASTER]: 1 nodes, 0 edges, 7 label blobs | REVIEW [LOW CONFIDENCE] | reasons=['too_few_nodes', 'too_few_edges', 'very_low_edge_node_ratio', 'node_segmentation_low_confidence', 'soft_raster_candidate_needs_graph_confirmation']
p354_soft_raster [SOFT RASTER]: 8 nodes, 4 edges, 4 label blobs | VALID
p364_raster: 4 nodes, 4 edges, 0 label blobs | REVIEW [LOW CONFIDENCE] | reasons=['node_segmentation_low_confidence']

Done.
O

In [20]:
# Cell 10: One-cell latency + validation summary + output file list

import pandas as pd
from pathlib import Path

total = (TIMING["end"] or time.perf_counter()) - (TIMING["start"] or time.perf_counter())

print("Pipeline summary")
print("=" * 70)
print("PDF:", PDF_PATH)
print("Output:", OUT_DIR)
print("Dry run:", DRY_RUN)
print("Total:", fmt(total))

print("\nStage times:")
stage_rows = [
    {"stage": k, "seconds": v, "time": fmt(v)}
    for k, v in TIMING["stages"].items()
]
df_stages = pd.DataFrame(stage_rows).sort_values("seconds", ascending=False) if stage_rows else pd.DataFrame()
display(df_stages)

print("\nPer-chart times + graph validation:")
chart_rows = []
for tag, info in TIMING["charts"].items():
    row = {"chart": tag}
    row.update(info)
    chart_rows.append(row)

df_charts = pd.DataFrame(chart_rows)

if len(df_charts):
    preferred_cols = [
        "chart", "page", "kind",
        "nodes", "edges", "edge_node_ratio",
        "is_valid_flowchart_graph", "needs_review", "validation_reasons",
        "confident",
        "nodes", "edges", "ocr", "total"
    ]

    # keep unique existing columns in preferred order, then append any remaining timing columns
    ordered = []
    for col in preferred_cols:
        if col in df_charts.columns and col not in ordered:
            ordered.append(col)
    ordered += [c for c in df_charts.columns if c not in ordered]

    df_charts = df_charts[ordered].sort_values(
        ["is_valid_flowchart_graph", "edges", "nodes"],
        ascending=[False, False, False]
    )

    validation_summary_path = os.path.join(OUT_DIR, "graph_validation_summary.csv")
    df_charts.to_csv(validation_summary_path, index=False)

    display(df_charts)
    print("\nValidation summary saved to:", validation_summary_path)
else:
    display(df_charts)

print("\nAccepted valid graph pages:")
if len(df_charts) and "is_valid_flowchart_graph" in df_charts.columns:
    valid_pages = df_charts[df_charts["is_valid_flowchart_graph"] == True]["page"].tolist()
    review_pages = df_charts[df_charts["needs_review"] == True]["page"].tolist()
    print("Valid:", valid_pages)
    print("Needs review:", review_pages)
else:
    print("No chart validation rows available.")

print("\nOutput files:")
for p in sorted(Path(OUT_DIR).glob("*")):
    if p.is_file():
        print(p)


Pipeline summary
PDF: /kaggle/input/datasets/boopaland3v/sam-data/The_European_Medical_Device_Regulation-SECURED.pdf
Output: /kaggle/working/mdr_flowchart_build
Dry run: True
Total: 21.05s

Stage times:


,stage,seconds,time
0,detect_charts,6.515710,6.52s
1,load_ocr,0.000014,0.00s



Per-chart times + graph validation:


,chart,page,kind,nodes,edges,edge_node_ratio,is_valid_flowchart_graph,needs_review,validation_reasons,confident,ocr,total
1,p015_raster,15,raster,23,20,0.8696,True,False,,True,0.149898,2.960231
3,p017_raster,17,raster,20,19,0.9500,True,False,,True,0.094638,1.672711
0,p014_vector,14,vector,18,19,1.0556,True,False,,True,0.200063,6.977777
2,p016_raster,16,raster,17,17,1.0000,True,False,,True,0.103526,2.150979
7,p354_soft_raster,354,soft_raster,8,4,0.5000,True,False,,True,0.127783,0.214048
8,p364_raster,364,raster,4,4,1.0000,False,True,node_segmentation_low_confidence,False,0.023131,0.109585
4,p275_soft_raster,275,soft_raster,3,1,0.3333,False,True,"too_few_edges, soft_raster_candidate_needs_gra...",True,0.038595,0.106667
5,p336_soft_raster,336,soft_raster,3,0,0.0000,False,True,"too_few_edges, very_low_edge_node_ratio, soft_...",True,0.052045,0.147534
6,p352_soft_raster,352,soft_raster,1,0,0.0000,False,True,"too_few_nodes, too_few_edges, very_low_edge_no...",False,0.104236,0.172217



Validation summary saved to: /kaggle/working/mdr_flowchart_build/graph_validation_summary.csv

Accepted valid graph pages:
Valid: [15, 17, 14, 16, 354]
Needs review: [364, 275, 336, 352]

Output files:
/kaggle/working/mdr_flowchart_build/all_graphs.json
/kaggle/working/mdr_flowchart_build/debug_p013_soft_raster.png
/kaggle/working/mdr_flowchart_build/debug_p014_vector.png
/kaggle/working/mdr_flowchart_build/debug_p015_raster.png
/kaggle/working/mdr_flowchart_build/debug_p016_raster.png
/kaggle/working/mdr_flowchart_build/debug_p017_raster.png
/kaggle/working/mdr_flowchart_build/debug_p275_soft_raster.png
/kaggle/working/mdr_flowchart_build/debug_p336_soft_raster.png
/kaggle/working/mdr_flowchart_build/debug_p352_soft_raster.png
/kaggle/working/mdr_flowchart_build/debug_p354_soft_raster.png
/kaggle/working/mdr_flowchart_build/debug_p364_raster.png
/kaggle/working/mdr_flowchart_build/graph_validation_summary.csv
/kaggle/working/mdr_flowchart_build/p013_soft_raster.json
/kaggle/working/m